### 1. Setup

In [20]:
import ollama
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END


class ChatState(TypedDict):
    user_input: str
    classification: str # 'technical' or 'casual'
    response: str

### 2. Define the Nodes

In [21]:
MODEL = "gemma3:1b"

def router_node(state: ChatState):
    print("--- ROUTING INTENT ---")
    prompt = f""""Classify the user message as 'TECHNICAL' (questions about code, science, math) or 'CASUAL' (greetings, small talk).
    Return only the word.
    Message: {state['user_input']}"""

    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    category = res['message']['content'].strip().upper()

    classification = 'technical' if "TECHNICAL" in category else 'casual'
    return {"classification": classification}

def technical_assistant(state: ChatState):
    print(" --- HANDLING TECHNICAL QUERY ---")
    prompt = f"You are a Senior Engineer. Provide a concise, professional answer to: {state['user_input']}"
    res = get_assistant_response('user', prompt)
    return res

def casual_assistant(state: ChatState):
    print("--- HANDLING CASUAL QUERY ---")
    prompt = f"You are a friendly friend. Reply warmly to: {state['user_input']}"
    res = get_assistant_response('user', prompt)
    return res

def get_assistant_response(role:str, prompt: str):
    res = ollama.chat(model=MODEL, messages=[{"role": role, "content": prompt}])
    return {"response": res['message']['content'].strip()}

### 3. Define the Graph Logic

In [22]:
# todo-SH: 1.Understand how StateGraph works, 2. What is Literal
def router_decision(state: ChatState) -> Literal["tech", "chat"]:
    if state['classification'] == 'technical':
        return "tech"
    else:
        return "chat"

# todo-SH: Don't know the below line
workflow = StateGraph(ChatState)

# Add our specialized nodes
workflow.add_node("router", router_node)
workflow.add_node("technical_handler", technical_assistant)
workflow.add_node("casual_handler", casual_assistant)

# Set the entry point
workflow.set_entry_point("router")

# Add conditional routing
workflow.add_conditional_edges(
    "router",
    router_decision,
    {
        "tech": "technical_handler",
        "chat": "casual_handler"
    }
)

# Both handlers go to END
workflow.add_edge("technical_handler", END)
workflow.add_edge("casual_handler", END)

app = workflow.compile()

### 4. Run the POC

In [23]:
# Test 1: Technical
result = app.invoke({"user_input": "How do i reverse a linked list in Python?"})
print(f"\n[AI]: {result['response']}")

# Test 2: Casual
result = app.invoke({"user_input": "Hey there! How's your day going?"})
print(f"\n[AI]: {result['response']}")

--- ROUTING INTENT ---
Response: model='gemma3:1b' created_at='2026-01-06T08:02:12.207378Z' done=True done_reason='stop' total_duration=652567917 load_duration=189196542 prompt_eval_count=61 prompt_eval_duration=417981791 eval_count=3 eval_duration=31732959 message=Message(role='assistant', content='TECHNICAL\n', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
 --- HANDLING TECHNICAL QUERY ---

[AI]: Okay, here's a concise and professional answer to how to reverse a linked list in Python, suitable for a senior engineer:

"There are several ways to reverse a linked list in Python. The most common and efficient method utilizes the `collections.deque` data structure.  It provides O(1) time complexity for both insertion and deletion at the beginning/end, making it ideal for this task.  Alternatively, you can manually traverse and construct a reversed list using a stack or another iterative approach, but `deque` offers superior performance."

**Key takeaways for a

### 5. Ask anything (REGRESSION TEST)

In [24]:
query = "What is the time complexity of quicksort?"
result = app.invoke({"user_input": query})
print(f"\n[AI]: {result['response']}")

--- ROUTING INTENT ---
Response: model='gemma3:1b' created_at='2026-01-06T08:02:17.47381Z' done=True done_reason='stop' total_duration=256653375 load_duration=168978709 prompt_eval_count=60 prompt_eval_duration=56030666 eval_count=3 eval_duration=29865042 message=Message(role='assistant', content='TECHNICAL\n', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
 --- HANDLING TECHNICAL QUERY ---

[AI]: Okay, here's a concise answer to the question, framed as a Senior Engineer:

"QuickSort's time complexity is generally O(n log n) on average, and O(n^2) in the worst case.  This stems from the recursive nature of the algorithm, which repeatedly divides the problem space and sorts sub-problems. While it performs well in practice, its worst-case scenario necessitates careful consideration of input data."

**To further enhance this answer, I'd add:**

*   **Mention the average case:**  "However, the average-case performance is highly dependent on the distribution of t

### 6. FAQ